# 플랜 생성기(days) SFT 합성 파이프라인 — 한 눈에 보기

사용자 발화 → **plan_generator 노드의 days 계약** SFT 데이터를 만드는 전 과정을
위에서 아래로 한 번 실행하면 끝나게 구성했다. **정보처리기사를 포함한 시험 6종**과
**OPIc(말하기)** 을 동시에 합성하고, 합치기·분할·검증까지 한다.

## 두 개의 트랙
| 트랙 | 언제 | 콘텐츠 출처 | 예시 |
|---|---|---|---|
| **template** (exam-synth) | 기출-반복형 시험(필기/실기) | 공용 `templates.build_plan`(개념→기출→오답→점검) | 정처기필기·토익·한능검·SQLD·컴활1/2급 |
| **domain** (crawl/curated) | 말하기·실기 등 도메인 특화 흐름 | 시험별 전용 task pool | OPIc(설문→오픽노잼→모의고사→실전) |

둘 다 **같은 days 계약**(`{summary_text, days:[{date, tasks:[{title, due_date}]}], personalization_patch}`)으로
출력하고, `system`/`user`/`parsed_goal` 은 런타임 미러(`plan_generator_template`)를 공유한다 → **학습==서빙**.
차이는 오직 `days` 의 **task 콘텐츠**뿐이다.

```
  SOURCES(레지스트리)
    +- exam-synth - build_exam_synth_plan_sft ------+
    +- opic - raw_cases->structure->build_opic -----+-> concat -> mix(release) -> split -> validate -> train
```
> 새 시험 추가 = **SOURCES 에 한 항목**(또는 exam_synth 에 한 줄). 맨 아래 §확장성 참고.


## 0. 환경 셋업 (CPU/macOS 에서 그대로 실행됨, GPU 불필요)

In [ ]:
import subprocess, sys, json
from pathlib import Path

# 노트북이 sft_pipeline/ 안에 있다고 가정하고 repo 루트를 찾는다.
ROOT = Path.cwd()
while ROOT.name != 'sft_pipeline' and ROOT != ROOT.parent:
    if (ROOT / 'sft_pipeline').exists():
        ROOT = ROOT / 'sft_pipeline'; break
    ROOT = ROOT.parent
REPO = ROOT.parent                      # mongle-ai (모든 -m 명령은 여기서 실행)
GEN  = 'sft_pipeline/data/generated'    # REPO 기준 상대경로
PY   = sys.executable

TODAY = '2026-06-25'   # due_date 기준일(재현 빌드 시 고정)
TOTAL = 600            # exam-synth 총 시드 수(6종 균등 분배)

def run(cmd: str):
    '''REPO 루트에서 명령 실행. 실패 시 stderr 출력 후 중단.'''
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=REPO, text=True, capture_output=True)
    if r.stdout.strip(): print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr.strip()[-1500:]); raise SystemExit(f'FAILED: {cmd}')

def load_jsonl(rel: str):
    return [json.loads(l) for l in (REPO / rel).read_text(encoding='utf-8').splitlines() if l.strip()]

print('REPO :', REPO)
print('GEN  :', REPO / GEN)


## 1. 소스 레지스트리 — **확장 지점**

각 소스는 `steps`(생성 명령들)와 최종 산출물 `out`, 출처 `provenance` 를 가진다.
**새 시험/도메인을 추가하려면 여기에 dict 하나만 더한다.** (자세한 방법은 맨 아래 §확장성)


In [ ]:
SOURCES = [
    {
        'name': 'exam-synth',          # 기출-반복형 6종(정처기·토익·한능검·SQLD·컴활1/2급)
        'track': 'template',
        'provenance': 'exam-synth',    # 공개 허용(원문 인용 없음)
        'out': f'{GEN}/exam_synth_sft.jsonl',
        'steps': [
            f'{PY} -m sft_pipeline.build.jobs.build_exam_synth_plan_sft '
            f'{GEN}/exam_synth_sft.jsonl --total {TOTAL} --today {TODAY}',
        ],
    },
    {
        'name': 'opic',                # 말하기 - 도메인 특화 task 흐름
        'track': 'domain',
        'provenance': 'exam-crawl',    # 블로그 기반 -> internal 전용(공개판 제외)
        'out': f'{GEN}/exam_opic_sft.jsonl',
        'steps': [
            f'{PY} sft_pipeline/build/jobs/build_raw_cases_opic.py',
            f'{PY} -m sft_pipeline.structure.run_structure '
            f'--in {GEN}/raw_cases_opic.csv --out {GEN}/structured_opic.csv',
            f'{PY} -m sft_pipeline.build.jobs.build_opic_plan_sft '
            f'{GEN}/structured_opic.csv {GEN}/exam_opic_sft.jsonl --today {TODAY}',
        ],
    },
    # v 새 소스는 여기에 추가 (예시는 맨 아래 §확장성)
]
[(s['name'], s['track'], s['provenance']) for s in SOURCES]


## 2. 각 소스 생성

In [ ]:
for s in SOURCES:
    print(f"\n=== [{s['name']}] ({s['track']}) ===")
    for step in s['steps']:
        run(step)


## 3. 소스별 점검 - days 계약 + 콘텐츠
런타임이 파싱하는 계약(<=30일·하루 1~3·전체 <=15·`due_date==date`·`title<=20`·difficulty/rationale 없음)을 전수 확인.

In [ ]:
from datetime import date, timedelta
t0 = date.fromisoformat(TODAY)

def check_contract(rows):
    for r in rows:
        a = json.loads(r['messages'][-1]['content'])
        assert list(a) == ['summary_text', 'days', 'personalization_patch']
        days = a['days']; ds = [d['date'] for d in days]
        assert ds == sorted(ds) and len(ds) == len(set(ds)) and len(days) <= 30
        assert sum(len(d['tasks']) for d in days) <= 15
        for d in days:
            assert 1 <= len(d['tasks']) <= 3
            assert t0 <= date.fromisoformat(d['date']) <= t0 + timedelta(days=29)
            for tk in d['tasks']:
                assert set(tk) == {'title', 'due_date'} and tk['due_date'] == d['date']
                assert 1 <= len(tk['title']) <= 20

for s in SOURCES:
    rows = load_jsonl(s['out'])
    check_contract(rows)
    a = json.loads(rows[0]['messages'][-1]['content'])
    print(f"[{s['name']}] {len(rows)}건  계약 OK")
    print('   예시 titles:', [tk['title'] for d in a['days'] for tk in d['tasks']][:5])
    print('   예시 summary:', a['summary_text'][:80])


## 4. 합치기 -> 분할 -> 검증
모든 소스 산출물을 모아 `mix_dataset` 로 **release 정책**(release 는 각 샘플의 `meta.provenance` 로
결정 - public 은 exam-crawl 제외)을 적용하고, `split_dataset` 로 provenance stratified 분할한다.

In [ ]:
RELEASE = 'internal'   # internal=전체 / public=exam-crawl(OPIc) 제외

# 모든 소스 산출물을 하나로 모은다(provenance 는 각 줄 meta 에 있어 mix 가 알아서 분류).
combined = f'{GEN}/plan_combined.jsonl'
with (REPO / combined).open('w', encoding='utf-8') as out:
    for s in SOURCES:
        out.write((REPO / s['out']).read_text(encoding='utf-8'))

run(f'{PY} -m sft_pipeline.build.lib.mix_dataset --exam-synth {combined} '
    f'--release {RELEASE} --out {GEN}/exam_mixed.jsonl')
run(f'{PY} -m sft_pipeline.build.lib.split_dataset --in {GEN}/exam_mixed.jsonl '
    f'--out-train {GEN}/sft_train.jsonl --out-valid {GEN}/sft_valid.jsonl')
run(f'{PY} -m sft_pipeline.build.lib.validate_dataset --in {GEN}/sft_train.jsonl')
run(f'{PY} -m sft_pipeline.build.lib.validate_dataset --in {GEN}/sft_valid.jsonl')


### 분포 확인

In [ ]:
import collections
for name in ('sft_train', 'sft_valid'):
    rows = load_jsonl(f'{GEN}/{name}.jsonl')
    by_prov = collections.Counter(r['meta'].get('provenance') for r in rows)
    by_exam = collections.Counter(r['meta'].get('exam_type') for r in rows)
    print(f'{name}: {len(rows)}건  provenance={dict(by_prov)}')
    print(f'   exam_type={dict(by_exam)}')


## 5. 학습 (RunPod GPU)
위 산출물은 `data/generated/`(gitignore)에 있다. RunPod 박스에선 이 노트북 §0~4 를
그대로 재실행해 재생성한 뒤(코드는 git 으로 동기화), 아래로 학습한다.

```bash
TRAIN=sft_pipeline/data/generated/sft_train.jsonl \
VALID=sft_pipeline/data/generated/sft_valid.jsonl \
OUT=outputs/exam-planner EPOCHS=1.0 \
bash sft_pipeline/train/runpod_dryrun.sh
```
> 템플릿형 소량 데이터라 **과적합이 더 큰 위험** -> 1 epoch 시작, train/eval loss 곡선으로 2 epoch 여부 판단.
> 품질의 레버는 epoch 가 아니라 **데이터 다양성**(exam_synth `--use-llm` 재서술, 도메인 확장).


## 6. 확장성 - 새 시험/도메인 추가하는 2가지 방법

### A) template 트랙 - 기출-반복형 시험 (가장 쉬움)
필기/실기처럼 '개념->기출->오답->점검' 흐름이 맞는 시험. **코드 한 줄도 안 늘리고** 두 곳만 편집:
1. `build/lib/exam_synth.py` 의 `EXAM_GOALS` 에 `"새시험": ["목표1", "목표2", "목표3"]` 추가
2. 같은 파일 `EXAM_STRATEGY` 에 `"새시험": "이 시험 공략 전략 한 문장"` 추가
3. (선택) `config/exam_types.yaml` 에 표준코드·별칭 등록(크롤/구조화에서 매칭)

-> `build_exam_synth_plan_sft` 가 자동으로 픽업(6종->7종). SOURCES 도 그대로.

### B) domain 트랙 - 말하기·실기 등 특화 흐름 (OPIc 패턴)
공용 템플릿의 '기출' 흐름이 **안 맞는** 시험(말하기·면접·포트폴리오 등). OPIc 생성기를 복사:
1. `build/jobs/build_opic_plan_sft.py` -> `build_<시험>_plan_sft.py` 복사
2. `_prep_pool`(도메인 task 순서)·`build_summary`(말투) 만 교체
   - `system`/`user`/`parsed_goal`/계약 검증은 공용 미러 그대로 두면 **학습==서빙 자동 유지**
3. 입력(raw_cases)은 크롤(`crawl/`) 또는 수기 큐레이션(`build_raw_cases_*.py`)
4. **SOURCES 에 항목 추가**:
```python
{
  'name': '토스', 'track': 'domain', 'provenance': 'exam-crawl',
  'out': f'{GEN}/exam_toeicspeaking_sft.jsonl',
  'steps': [f'{PY} -m sft_pipeline.build.jobs.build_toeicspeaking_plan_sft ... --today {TODAY}'],
}
```

### 어느 트랙인지 고르는 기준
> task 제목이 **'기출 N회차/개념/오답'으로 말이 되면 A**, 그렇지 않고 **도메인 고유 행동**(설문 선택·모의 발화·작품 제출)이 필요하면 **B**.

### 공통 규약 (어느 트랙이든)
- 출력은 **days 계약**, `meta.node='plan_generator'`(검증 라우팅), `meta.today` 필수
- `provenance`: 합성=**exam-synth**(공개 허용) / 크롤=**exam-crawl**(internal 전용)
- 일상(운동·여행·루틴) 트랙도 같은 구조로 추가 가능(daily 생성기는 별도 - 향후 C)
